Thoughts:
- llama and command r can decently simplify sentences zero-shot.
- add HSK vocabulary context so simplification includes more HSK1-4 vocab.
- add user-specific vocab knowledge which gets added to context. This will be complex words that the user knows (don't simplify) or complex words the user doesn't know (if it is a key word to the meaning, keep it, otherwise, simplify it)
- optional end-goals/simplification level: (1) extensive reading, keep it as simple as possible (2) a little intensive so the user can learn more words

Notebook for experimenting with bedrock LLM for text simplification

In [1]:
%load_ext autoreload
%autoreload 2
import boto3
from botocore.exceptions import ClientError
from langchain_aws.llms.bedrock import BedrockLLM
from langchain_aws.chat_models import ChatBedrockConverse
import utils.bedrock_pipeline as bp
import utils.evaluate as eval
import re
from bs4 import BeautifulSoup

c:\Users\tempu\Documents\Code\text_simplification\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Create an Amazon Bedrock Runtime client.
brt = boto3.client("bedrock-runtime")

# Set the model ID
arn_cr = "cohere.command-r-v1:0"
arn_ds = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.deepseek.r1-v1:0"
arn_llama = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-2-3b-instruct-v1:0"
arn_llama2 = "arn:aws:bedrock:us-east-1:711387109035:inference-profile/us.meta.llama3-3-70b-instruct-v1:0"

llm_llama = ChatBedrockConverse(client=brt,
                        model_id=arn_llama,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=4000
                        ,)

llm_llama2 = ChatBedrockConverse(client=brt,
                        model_id=arn_llama2,
                        provider="meta",
                        temperature=0.1,
                        max_tokens=8000
                        ,)

llm_cr = ChatBedrockConverse(client=brt,
                        model=arn_cr,
                        temperature=0.1,
                        max_tokens=50,)

In [3]:
llm = llm_llama2

### Evaluation pipeline

In [12]:
sentences = eval.mcts[0:200]
simplification_prompt = bp.build_simplification_prompt_oneshot(sentences)
print(simplification_prompt)

请简化下面这段中文，使每个句子的用词更简单，适合中文学习者阅读。请特别注意以下几点：

1. 替换生僻词、高级词汇或抽象表达，使用更常见、更直白的词语或短语。
2. 避免使用超出HSK6级范围的词汇，优先使用HSK 1-3级中常见词汇。
3. 保留原句数量和顺序，只进行词语层面的简化，不要省略、合并或总结内容。
4. 只输出简化后的句子，不要解释或分析。
5. 确保简化句子的语义与原句子保持一致。

例如：
    原文段落：凌晨五点到达云南丽江，走在古老的街道上，身边空无一人。寒风凛冽，天寒地冻，远山高大寂寥的轮廓在茫茫晨雾中依稀可见。李晟走了很久，才看到孟北晗的身影从雾气尽头慢慢浮现。
    简化后的段落：李晟早上五点到达云南丽江，走在老街上，周围一个人都没有。风很冷，李晟走了很久，才看到孟北晗从雾中走出来。

原文段落：
0. 一、在中华人民共和国领土内及中华人民共和国注册的运输工具上就业或者工作的最低年龄为十六周岁；

1. 北京国际图书博览会是一九八六年经国务院批准由中国图书进出口总公司举办的，每两年举办一届。

2. 根据该协议，印度在三年内取消进口斯里兰卡商品的所有关税，斯里兰卡则在八年内取消进口印度商品的关税。

3. 斯里兰卡总统库马拉通加夫人是２７日下午抵达这里，开始对印度进行为期三天的正式友好访问的。

4. 在施工过程中，加拿大咨询专家发现由于地质情况比较好，而且隧洞是明流洞，洞顶拱不过流，因此建议取消顶拱的衬砌。

5. 卫生部已确定在未来几年内实施一项艾滋病防治的公众教育计划，以进一步提高人们认识，增强公众自我保护能力。

6. 去年，卫生部和有关部门提出了“预防为主，宣传教育为主，经常性工作为主”的艾滋病防治策略，并得到国务院的确认。

7. 中国又一条煤炭运输大通道──连接天津蓟县与天津港之间的蓟港铁路日前破土动工。

8. 这位官员说，为了适应国民经济对石油日益增长的需求，今后中国的石油发展战略，将继续立足国内资源，加快陆上和海上石油、天然气的勘探开发，努力提高油气产量。

9. 蒙古人酗酒现象日趋严重，每年被强制醒酒的人达１０万之众，酗酒已成为滋生犯罪的主要根源之一。

10. 这位官员说，煤层气是具有很大开发潜力的“绿色能源”，已被列为中国新能源发展战略的重点之一。

11. 届时如能放手一搏，充分发挥自己的水平，常、周两人同

In [13]:
simplified_raw = llm.invoke(simplification_prompt)
simplified = bp.parse_llm_output(simplified_raw.content)
print(simplified)

['在中国境内或者中国的交通工具上工作的最低年龄是十六周岁。', '北京国际图书博览会是1986年开始的，每两年举办一次。', '根据协议，印度将在三年内取消对斯里兰卡商品的关税，斯里兰卡则在八年内取消对印度商品的关税。', '斯里兰卡总统库马拉通加夫人27日下午抵达这里，开始对印度进行为期三天的正式友好访问。', '在施工过程中，加拿大专家发现地质情况很好，建议取消隧道顶部的衬砌。', '卫生部将实施一项艾滋病防治的公众教育计划，以提高人们的认识和自我保护能力。', '去年，卫生部提出了“预防为主”的艾滋病防治策略，并得到国务院的确认。', '连接天津蓟县与天津港之间的铁路日前开始建设。', '中国的石油发展战略将继续立足国内资源，加快石油和天然气的勘探开发。', '蒙古人酗酒现象很严重，每年有10万人被强制醒酒，酗酒已成为滋生犯罪的主要根源之一。', '煤层气是具有很大开发潜力的“绿色能源”，已被列为中国新能源发展战略的重点之一。', '如果常、周两人发挥自己的水平，闯入四强也不算是非分之想。', '一些国家决定从明年起给予最不发达国家出口优惠待遇。', '天津港保税区1997年全面超额完成各项经济指标，平均增幅达70%以上。', '广西的十七条航线连接了国内的各大城市，极大地方便了交通。', '人民日报刊载了中国人民银行副行长高德柱关于中国金融改革的发言。', '要建立一个金融工具多样化、依法管理的金融市场体系。', '外国企业纷纷到美国证券交易市场上市，因为美国经济稳定增长，纽约证交所管理严格。', '如果外国企业没有条件上市，不要急于到纽约上市，否则难免受到冷落。', '尼日利亚国家元首阿布巴卡尔指出，1998年尼日利亚石油出口收入锐减，财政状况恶化。', '由于石油收入锐减，尼政府财政入不敷出，10月份不得不动用外汇储备18.7亿美元。', '阿布巴卡尔说，国际原油市场目前仍无好转迹象，尼石油收入不大可能大幅增加。', '电视等媒体的介入，使体育运动成为全世界同时关注的事情。', '这些经纪人认为，资方部署了一个阴谋，故意让赛季死亡，然后再重新获得控制权。', '美英对伊发动的侵略违反了联合国宪章和国际法准则。', '阿齐兹强调，现在是阿拉伯各国政府起来谴责美英所犯下的滔天罪行的时候了。', '台湾经济学家侯家驹指出，台湾当局对台商投资大陆采取“戒急用忍

In [6]:
judge_prompt = bp.build_judge_prompt(sentences, simplified)
print(judge_prompt)

请对下列两个句子进行对比，判断简化句是否忠实保留了原句的语义，且语言通顺、自然。

如果简化句与原句意义相符且语言流畅，请直接输出简化句；
如果简化句表达不准确或不够自然，请你重新改写，使其既保留原句的主要信息，又更加容易被中文学习者理解。

0. 原句：[一、在中华人民共和国领土内及中华人民共和国注册的运输工具上就业或者工作的最低年龄为十六周岁；
], 简化句：[在中国境内或者中国的交通工具上工作的最低年龄是十六岁。]
1. 原句：[北京国际图书博览会是一九八六年经国务院批准由中国图书进出口总公司举办的，每两年举办一届。
], 简化句：[北京国际图书博览会是1986年开始的，每两年举办一次。]
2. 原句：[根据该协议，印度在三年内取消进口斯里兰卡商品的所有关税，斯里兰卡则在八年内取消进口印度商品的关税。
], 简化句：[根据协议，印度将在三年内取消对斯里兰卡商品的关税，斯里兰卡则在八年内取消对印度商品的关税。]
3. 原句：[斯里兰卡总统库马拉通加夫人是２７日下午抵达这里，开始对印度进行为期三天的正式友好访问的。
], 简化句：[斯里兰卡总统库马拉通加夫人27日下午抵达这里，开始对印度进行三天的正式友好访问。]
4. 原句：[在施工过程中，加拿大咨询专家发现由于地质情况比较好，而且隧洞是明流洞，洞顶拱不过流，因此建议取消顶拱的衬砌。
], 简化句：[在施工过程中，加拿大专家发现地质情况比较好，建议取消隧道顶部的衬砌。]
5. 原句：[卫生部已确定在未来几年内实施一项艾滋病防治的公众教育计划，以进一步提高人们认识，增强公众自我保护能力。
], 简化句：[卫生部将实施艾滋病防治的公众教育计划，以提高人们的认识和自我保护能力。]
6. 原句：[去年，卫生部和有关部门提出了“预防为主，宣传教育为主，经常性工作为主”的艾滋病防治策略，并得到国务院的确认。
], 简化句：[去年，卫生部提出了“预防为主”的艾滋病防治策略，并得到国务院的确认。]
7. 原句：[中国又一条煤炭运输大通道──连接天津蓟县与天津港之间的蓟港铁路日前破土动工。
], 简化句：[连接天津蓟县和天津港的铁路日前开始建设。]
8. 原句：[这位官员说，为了适应国民经济对石油日益增长的需求，今后中国的石油发展战略，将继续立足国内资源，加快陆上和海上石油、天然气的勘探开发，努力提高油气产量。
], 简化句：[中

In [7]:
judged_raw = llm.invoke(judge_prompt)
judged = bp.parse_llm_output(judged_raw.content)
print(judged)

['在中国境内或者中国的交通工具上工作的最低年龄是十六岁。', '北京国际图书博览会是1986年开始的，每两年举办一次。', '根据协议，印度将在三年内取消对斯里兰卡商品的关税，斯里兰卡则在八年内取消对印度商品的关税。', '斯里兰卡总统库马拉通加夫人27日下午抵达这里，开始对印度进行三天的正式友好访问。', '在施工过程中，加拿大专家发现地质情况比较好，建议取消隧道顶部的衬砌。', '卫生部将实施艾滋病防治的公众教育计划，以提高人们的认识和自我保护能力。', '去年，卫生部提出了“预防为主”的艾滋病防治策略，并得到国务院的确认。', '连接天津蓟县和天津港的铁路日前开始建设。', '中国的石油发展战略将继续立足国内资源，加快石油和天然气的勘探开发。', '蒙古人酗酒现象严重，每年有10万人被强制醒酒，酗酒已成为滋生犯罪的主要根源之一。', '煤层气是具有很大开发潜力的“绿色能源”，已被列为中国新能源发展战略的重点之一。', '如果两人发挥自己的水平，同時闯入四强也不算是非分之想。', '一些国家决定从明年起给予最不发达国家出口优惠待遇。', '天津港保税区1997年全面超额完成各项经济指标，平均增幅达70%以上。', '广西的十七条航线连接了沿海和国内的各大城市，极大地方便了交通。', '人民日报刊载了中国人民银行副行长高德柱关于中国金融改革的发言。', '要建立一个金融工具多样化、依法管理、有序竞争的金融市场体系。', '外国企业纷纷到美国证券交易市场上市，因为美国经济稳定增长，纽约证交所管理严格。', '如果外国企业尚无条件到纽约上市，不要盲目急于上市，否则难免受到冷落。', '尼日利亚国家元首阿布巴卡尔指出，1998年尼石油出口收入锐减，财政状况恶化。', '由于石油收入锐减，尼政府财政入不敷出，10月份不得不动用外汇储备18.7亿美元。', '阿布巴卡尔说，国际原油市场目前仍无好转迹象，尼石油收入不大可能大幅增加。', '电视等媒体的介入，使体育运动成为全世界同时关注的事情，也使新闻媒体获得巨额收入。', '经纪人认为，资方部署了一个阴谋，故意让赛季死亡，然后再重新获得控制权。', '美英对伊发动的侵略违反了联合国宪章和国际法准则，是对联合国安理会权威的藐视。', '阿齐兹强调，现在是阿拉伯各国政府起来谴责美英所犯下的滔天罪行的时候了。', '台湾经济学家

In [ ]:
print(len(judged), len(sentences[0:100]))

In [14]:
print(len(simplified), len(sentences[0:100]))

100 100


In [15]:
l13, freq = eval.corpus_metrics(sentences[0:100], simplified)
sari, bleu, f1 = eval.saribleu_metrics(simplified, eval.mcts_ref[0][0:100], eval.mcts[0:100])

In [16]:
print(l13, freq, sari, bleu, f1)

-3.9545205937604027 0.9177008978193836 40.299888211470396 0.26270429641692244 0.9657760870456695


In [17]:
import csv   
fields=["L1-3 oneshot without judge", f"{l13:.3f}", f"{freq:.3f}", f"{sari:.3f}", f"{bleu:.3f}", f"{f1:.3f}"]
with open(r"../data/results/llm_results.csv", 'a') as f:
    writer = csv.writer(f)
    writer.writerow(fields)

### Web reading pipeline

In [ ]:
import requests
import re
from bs4 import BeautifulSoup
url = 'https://read.douban.com/reader/column/69884011/chapter/622526435'
site = requests.get(url)
site.encoding = 'utf-8'
site_soup = BeautifulSoup(site.text, 'html.parser')

In [4]:
url = 'https://read.douban.com/reader/column/69884011/chapter/622526435'
soup = bp.extract_from_url(url)

In [ ]:
### For extracting sentences from <p> tags, while preserving quotations
sentences = []
for element in soup.find_all('p'):
    # print("Next:", element.decode_contents())
    paragraph = element.get_text(separator='', strip=True)  # Extract plain text without tags
    parts = re.split(r'([。！？!?．][”’」』"]?)', paragraph)
    
    # Recombine the split pieces into full sentences
    for i in range(0, len(parts) - 1, 2):
        sentence = parts[i] + parts[i + 1]
        sentence = sentence.strip()
        if sentence:
            sentences.append(sentence)

    # Handle any trailing part without punctuation
    if len(parts) % 2 == 1 and parts[-1].strip():
        sentences.append(parts[-1].strip())

In [7]:
### For extracting all text from <p> tags (multiple sentences)
sentences = []
for element in soup.find_all('p'):
    # print("Next:", element.decode_contents())
    paragraph = element.get_text(separator='', strip=True)  # Extract plain text without tags
    text = re.split(r'(?<=[。！？!?．])\s*', paragraph)
    sentences.append(paragraph)

In [8]:
len(sentences), sentences[:5]

(166,
 ['2021年初冬，李晟最后一次见到孟北晗。',
  '凌晨五点到达云南丽江，走在古老的街道上，身边空无一人。寒风凛冽，天寒地冻，远山高大寂寥的轮廓在茫茫晨雾中依稀可见。李晟走了很久，才看到孟北晗的身影从雾气尽头慢慢浮现。',
  '孟北晗那头栗色长卷发没来得及打理，散乱地披在身后，抱歉地说：“小乐生病了，我刚从医院赶过来。”',
  '李晟立刻紧张起来：“小乐怎么了？严不严重？”',
  '孟北晗摇摇头，目光柔和：“没事。可能是昨晚吃坏了肚子，凌晨时候吐了。医生说是急性胃肠炎，不碍事。”'])

In [10]:
simplification_prompt = bp.build_simplification_prompt(sentences)

In [11]:
simplified_raw = llm.invoke(simplification_prompt).content
simplified = bp.parse_llm_output(simplified_raw)

In [12]:
judge_prompt = bp.build_judge_prompt(sentences, simplified)

In [13]:
judged_raw = llm.invoke(judge_prompt).content
judged = bp.parse_llm_output(judged_raw)

In [ ]:
ner_prompt = bp.build_NER_prompt(judged)

In [39]:
ner_raw = llm.invoke(ner_prompt).content
ner = bp.parse_llm_output(ner_raw)

In [40]:
import webbrowser
html = bp.generate_html(url, ner)
new_soup = BeautifulSoup(html, features="html.parser") # parse to html

with open("output.html", "w", encoding="utf-8") as file:
    file.write(str(new_soup)) # write html to file
webbrowser.open("output.html");

In [17]:
bp.corpus_metrics(sentences[:len(judged)], judged)

(np.float64(-0.8429261020124641), np.float64(0.2587944686426238))